In [ ]:
%cd /home/ubuntu/datajoint_wahl
import seaborn as sns
from schema import mpanze_face_tracking as ft
from schema.mpanze_paw_tracking_refactor import mpanze_paw_tracking_refactor as pt
from schema.mpanze_widefield_refactor import mpanze_widefield_refactor as wf
from schema.mpanze_exp_refactor import mpanze_exp_refactor as exp

import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from statannotations.Annotator import Annotator, PValueFormat

import cv2
from mpanze_scripts.util.allen_utils import load_allen, overlay_allen


base = importr('base')
lme4 = importr('lme4')
emmeans = importr('emmeans')
stats = importr('stats')

from scipy.interpolate import interp1d
from scipy.stats import false_discovery_control
import pandas as pd
import tqdm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
from pathlib import Path
%matplotlib inline

# set font to Arial
rc('font',**{'family':'sans-serif','sans-serif':['Arial']})
# set font sizes to 12 for figures
rc('font', size=12)          # controls default text sizes
rc('axes', titlesize=12)     # fontsize of the axes title
rc('axes', labelsize=12)    # fontsize of the x and y labels
rc('xtick', labelsize=12)    # fontsize of the tick labels
rc('ytick', labelsize=12)    # fontsize of the tick labels
rc('legend', fontsize=10)    # legend fontsize
rc('figure', titlesize=12)  # fontsize of the figure title

# set line width to 1
rc('lines', linewidth=1)

# set dpi to 600 for figures
rc('figure', dpi=600)

# svg font type shenanigans
rc('svg', fonttype='none')

fontsize_small = 10
fontsize_medium = 12
fontsize_large = 14

# define conversion factor from inches to cm for convenience
cm = 1/2.54 * 1.5 # (scale larger for Manuscript)

# color palette for cohorts
group_colors = {'Sham':'#BBBBBB', 'Stroke':'#4477AA', 'Stroke + training':'#AA3377'}

p_figures = Path('~/neurophys_3/r_outputs/figures/rev_supplementary_pupil/').expanduser()
p_figures.mkdir(parents=True, exist_ok=True)

### Supplementary Figure 10 A

In [ ]:
# get keys for all sessions - excluding learning
keys = (
    ft.InterpolatedRadius
    * pt.RestingMask
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group="group")
    & "phase != 'Learning'"
    & [f"days_from_stroke_norm = {day}" for day in [-3,-2,-1,3,7,14,21,28]]
    & "stroke_group != 'Learning'"
).fetch("KEY", as_dict=True)
print(f"Found {len(keys)} expert sessions")

# fetch data
entries = []
for key in tqdm.tqdm(keys):
    try:
        # interpolated pupil radius (already aligned to wf timestamps)
        radius = (ft.InterpolatedRadius & key).fetch1('radius')

        # widefield timestamps
        t_wf = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")

        # paw timestamps and resting masks for both hands
        t_L = (pt.Synchronisation.Hand & dict(**key, hand='L')).fetch1("frame_timestamps")
        t_R = (pt.Synchronisation.Hand & dict(**key, hand='R')).fetch1("frame_timestamps")
        mask_L = (pt.RestingMask.Hand & dict(**key, hand='L')).fetch1("paw_still_mask")
        mask_R = (pt.RestingMask.Hand & dict(**key, hand='R')).fetch1("paw_still_mask")

        # interpolate masks to wf time base (nearest) and combine
        f_L = interp1d(t_L, mask_L.astype(float), kind='nearest', fill_value="extrapolate", bounds_error=False)
        f_R = interp1d(t_R, mask_R.astype(float), kind='nearest', fill_value="extrapolate", bounds_error=False)
        mask_L_interp = f_L(t_wf).astype(bool)
        mask_R_interp = f_R(t_wf).astype(bool)
        resting_mask = mask_L_interp & mask_R_interp

        radius_rest = np.nanmean(radius[resting_mask]) if resting_mask.any() else np.nan
        radius_active = np.nanmean(radius[~resting_mask]) if (~resting_mask).any() else np.nan

        # days_from_stroke_norm if present
        day = (exp.DaysFromStrokeNorm & key).fetch1('days_from_stroke_norm')

        entries.append(dict(**key, days_from_stroke_norm=day, radius_rest=radius_rest, radius_active=radius_active))
    except Exception as e:
        print(f"Skipping key {key} due to error: {e}")

df = pd.DataFrame(entries)
display(df.head())


# run stats
df_melt = pd.DataFrame(dict(
    m=np.concatenate([df["mouse_id"], df["mouse_id"]]),
    d=np.concatenate([df["days_from_stroke_norm"], df["days_from_stroke_norm"]]),
    s = ['rest'] * len(df) + ['active'] * len(df),
    r = np.concatenate([df['radius_rest'].to_numpy(), df['radius_active'].to_numpy()])                            
))

df_melt["s"] = pd.Categorical(df_melt["s"], ["rest", "active"])
df_melt["d_cat"] = pd.Categorical(df_melt["d"].astype(str), sorted(df_melt["d"].unique().astype(str), key=lambda x: float(x)))
display(df_melt)

with (robjects.default_converter + pandas2ri.converter).context():
        model = lme4.lmer('r ~ 1 + s + (1|m/d_cat)', data=df_melt)
        formula = "pairwise ~ s"
        emm = emmeans.emmeans(model, stats.formula(formula), adjust="none")
        contrasts = base.summary(emm[1])
        confints = stats.confint(emm[1])

display(contrasts)
mean_diff = contrasts.iloc[0]['estimate']
sem_diff = contrasts.iloc[0]['SE']
p_val = contrasts.iloc[0]['p.value']


# plot
f, ax = plt.subplots(1,1, figsize=(4.5,4.5))
sns.boxplot(data=df_melt, x='s', y='r', order=['rest','active'], boxprops=dict(facecolor='none'), showfliers=False, ax=ax, hue='s')
sns.stripplot(data=df_melt, x='s', y='r', order=['rest','active'], size=3, jitter=True, alpha=0.6, ax=ax, hue='s')

ax.set_xlabel('State')
ax.set_ylabel('Pupil radius (pixels)')

ax.text(
    0.5,
    0.88,
    f"difference (active - rest) = {-mean_diff:.3f} ± {sem_diff:.3f} pixels",
    ha="center",
    va="top",
    transform=ax.transAxes,
    fontsize=fontsize_small,
)
# annotate
ax.text(0.5, 0.95, f'Linear mixed-effects model {"p < 0.001" if p_val < 0.001 else f"p = {p_val:.3f}"}', ha='center', va='top', transform=ax.transAxes, fontsize=fontsize_small)
f.suptitle(f'Pupil radius: Rest vs Active (paired).\n n sessions = {len(df)}, n mice = {df["mouse_id"].nunique()}', fontsize=fontsize_medium)

plt.tight_layout()
display(f)
f.savefig(p_figures / 'pupil_radius_rest_vs_active_paired.svg', transparent=True, dpi=300)
f.savefig(p_figures / 'pupil_radius_rest_vs_active_paired.png', transparent=True, dpi=300)

# print caption for figure
caption = (
    f"Average pupil radius during active and resting states. Each datapoint represents an experimental session (n = {len(df)} sessions from n = {df['mouse_id'].nunique()} mice). All experimental sessions except for the learning sessions were included. Paired differences were assessed using a linear mixed-effects model, the results of which are reported in the panel."
)

caption_path = p_figures / 'pupil_radius_rest_vs_active_paired_caption.txt'
caption_path.write_text(caption + "\n", encoding='utf-8')

# print statistical table
stat_table = pd.DataFrame([{
    'Panel': "sf_revisions_pupil_A",
    'Statistical test': "Linear mixed-effects model",
    'Comparison': f"{contrasts.iloc[0]['contrast']} pupil radius",
    'Effect size': f"{mean_diff:.3f} pixels",
    '95% Confidence intervals': f"[{confints.iloc[0]['lower.CL']:.3f}, {confints.iloc[0]['upper.CL']:.3f}]",
    'P-value': f"{p_val:.3e}",
    'Post hoc test / multiple comparisons': "n/a",
    'Adj. P-value': "n/a",
}])
display(stat_table)
stat_table.to_csv(p_figures / 'pupil_radius_rest_vs_active_paired_stats.csv', index=False)

# raw data
df.filter(['mouse_id', 'days_from_stroke_norm', 'radius_rest', 'radius_active']).to_csv(p_figures / 'pupil_radius_rest_vs_active_paired_raw_data.csv', index=False)

# Supplementary figure 10 B 

In [ ]:
rois = [
    "MOs-lateral", "MOs-medial", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSp-anterior", "RSP-posterior", "VISp", "VIS-medial", "VISa", "VISrl",
]
rois_for_stats = [r + "_contra" for r in rois] + [r + "_ipsi" for r in rois]

wf_param_id = 6
phase_order = ["Expert", "Early", "Late"]

keys = (
    ft.InterpolatedRadius
    * pt.RestingMask
    * exp.ExperimentalPhase
    * exp.DaysFromStrokeNorm
    * exp.StrokeGroup.proj(stroke_group="group")
    * wf.ImageProcessing2
    & f"wf_param_id={wf_param_id}"
    & "mouse_id > 36"
    & "phase != 'Learning'"
    & "stroke_group != 'Learning'"
    & [f"days_from_stroke_norm = {day}" for day in [-3,-2,-1,3,7,14,21,28]]
).fetch("KEY", as_dict=True)
print(f"Found {len(keys)} sessions with pupil tracking and resting masks")


rows_maps = []
rows_rois = []
for key in tqdm.tqdm(keys, desc="Processing sessions"):
    # take pupil after 1 minute as there are early artifacts due to light adaptation
    u, svt, h, w = (wf.ImageProcessing2 & key).load_components(svt_baseline=True)
    svt = svt[:, 120:]
    t_wf = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")[120:]
    radius = (ft.InterpolatedRadius & key).fetch1("radius")[120:]
    M = (wf.RescaledAllenRegistration2 & key).fetch1("allen_matrix_rescaled")
    handedness = (exp.Handedness & key).fetch1("handedness")

    # compute resting mask
    t_L = (pt.Synchronisation.Hand & dict(**key, hand="L")).fetch1("frame_timestamps")
    t_R = (pt.Synchronisation.Hand & dict(**key, hand="R")).fetch1("frame_timestamps")
    mask_L = (pt.RestingMask.Hand & dict(**key, hand="L")).fetch1("paw_still_mask")
    mask_R = (pt.RestingMask.Hand & dict(**key, hand="R")).fetch1("paw_still_mask")
    f_L = interp1d(t_L, mask_L.astype(float), kind="nearest", fill_value="extrapolate", bounds_error=False)
    f_R = interp1d(t_R, mask_R.astype(float), kind="nearest", fill_value="extrapolate", bounds_error=False)
    resting_mask = f_L(t_wf).astype(bool) & f_R(t_wf).astype(bool)

    # compute high vs low masks
    radius_threshold = np.nanmedian(radius[resting_mask])
    low_mask = resting_mask & (radius < radius_threshold)
    high_mask = resting_mask & (radius >= radius_threshold)

    # compute low vs high maps
    svt_low = np.nanmean(svt[:, low_mask], axis=1) * 100
    svt_high = np.nanmean(svt[:, high_mask], axis=1) * 100
    map_low = (u @ svt_low).reshape(h, w).astype(np.float32)
    map_high = (u @ svt_high).reshape(h, w).astype(np.float32)
    map_low = cv2.warpAffine(map_low, M, (w, h))
    map_high = cv2.warpAffine(map_high, M, (w, h))
    if handedness == "R":
        map_low = np.fliplr(map_low)
        map_high = np.fliplr(map_high)

    rows_maps.append(
        dict(
            **key,
            phase=(exp.ExperimentalPhase & key).fetch1("phase"),
            days_from_stroke_norm=(exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm"),
            stroke_group=(exp.StrokeGroup.proj(stroke_group="group") & key).fetch1("stroke_group"),
            radius_threshold=float(radius_threshold),
            n_rest=int(resting_mask.sum()),
            n_low=int(low_mask.sum()),
            n_high=int(high_mask.sum()),
            n_total=len(resting_mask),
            low_map=map_low,
            high_map=map_high,
        )
    )

    # iterate over rois and extract mean values for low and high maps
    for roi in rois_for_stats:
        roi_dff = (wf.AllenSegmentation2.ROI & dict(**key, roi_id=roi)).fetch1("dff")[120:]
        roi_low = np.nanmean(roi_dff[low_mask]) * 100
        roi_high = np.nanmean(roi_dff[high_mask]) * 100
        # append twice
        rows_rois.append(
            dict(
                **key,
                roi=roi,
                phase=(exp.ExperimentalPhase & key).fetch1("phase"),
                days_from_stroke_norm=(exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm"),
                stroke_group=(exp.StrokeGroup.proj(stroke_group="group") & key).fetch1("stroke_group"),
                radius_threshold=float(radius_threshold),
                n_rest=int(resting_mask.sum()),
                n_low=int(low_mask.sum()),
                n_high=int(high_mask.sum()),
                n_total=len(resting_mask),
                state='low',
                response=roi_low,
            )
        )
        rows_rois.append(
            dict(
                **key,
                roi=roi,
                phase=(exp.ExperimentalPhase & key).fetch1("phase"),
                days_from_stroke_norm=(exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm"),
                stroke_group=(exp.StrokeGroup.proj(stroke_group="group") & key).fetch1("stroke_group"),
                radius_threshold=float(radius_threshold),
                n_rest=int(resting_mask.sum()),
                n_low=int(low_mask.sum()),
                n_high=int(high_mask.sum()),
                n_total=len(resting_mask),
                state='high',
                response=roi_high,
            )
        )

df_rest_pupil_maps = pd.DataFrame(rows_maps)
df_rest_pupil_rois = pd.DataFrame(rows_rois)
df_rest_pupil_maps.to_pickle(p_figures / f"rest_pupil_maps_wf{wf_param_id}.pkl")
df_rest_pupil_rois.to_pickle(p_figures / f"rest_pupil_rois_wf{wf_param_id}.pkl")
display(df_rest_pupil_maps.head())
display(df_rest_pupil_rois.head())

In [ ]:
wf_param_id = 6
df_rest_pupil_maps = pd.read_pickle(p_figures / f"rest_pupil_maps_wf{wf_param_id}.pkl")
df_rest_pupil_rois = pd.read_pickle(p_figures / f"rest_pupil_rois_wf{wf_param_id}.pkl")

df_rest_pupil_rois["p"] = pd.Categorical(df_rest_pupil_rois["phase"], categories=["Expert", "Early", "Late"], ordered=True)
df_rest_pupil_rois["s"] = pd.Categorical(df_rest_pupil_rois["state"], categories=["low", "high"], ordered=True)
df_rest_pupil_rois["r"] = df_rest_pupil_rois["response"]
df_rest_pupil_rois["d"] = pd.Categorical(df_rest_pupil_rois["days_from_stroke_norm"].astype(str), sorted(df_rest_pupil_rois["days_from_stroke_norm"].unique().astype(str), key=lambda x: float(x)))
df_rest_pupil_rois["m"] = df_rest_pupil_rois["mouse_id"]

df_rest_pupil_rois.head()

In [ ]:
areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl",
]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
#areas_to_dot = ["MOs-medial_R", "MOs-medial_L", "RSP-anterior_R", "RSP-anterior_L"]

masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# close holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
mask_combined = (mask_combined > 0).astype(np.uint8) * 255

stat_contrasts = []
for roi, df_roi in df_rest_pupil_rois.filter(["roi", "r", "s", "p", "d", "m"]).groupby("roi"):
    with (robjects.default_converter + pandas2ri.converter).context():
        model = lme4.lmer('r ~ 1 + s * p + (1|m/d)', data=df_roi)
        formula = "pairwise ~ s | p"
        emm = emmeans.emmeans(model, stats.formula(formula), adjust="none")
        contrasts = base.summary(emm[1])
        confints = stats.confint(emm[1])

    contrasts["roi"] = roi
    contrasts["confints"] = "[{:.3f}, {:.3f}]".format(confints.iloc[0]['lower.CL'], confints.iloc[0]['upper.CL'])
    confints["roi"] = roi
    stat_contrasts.append(contrasts)
    
stat_contrasts = pd.concat(stat_contrasts, ignore_index=True)
# false discovery rate correction for multiple comparisons
p_values = stat_contrasts["p.value"].values
p_adj = false_discovery_control(p_values, method='bh')
stat_contrasts["p_adj"] = p_adj
display(stat_contrasts)

nanmean = lambda x: np.nanmean(np.stack(x), axis=0)
df_for_plotting = (
    df_rest_pupil_maps
    .groupby(["phase", "mouse_id"])[["low_map", "high_map"]]
    .agg(nanmean)
    .groupby("phase")
    .agg(nanmean)
)
df_for_plotting["difference_map"] = df_for_plotting["high_map"] - df_for_plotting["low_map"]
display(df_for_plotting.head())

h,w = 128, 128

from scipy.ndimage import center_of_mass
phases = ["Expert", "Early", "Late"]
plt.close("all")
f, ax = plt.subplots(3,3, figsize=(12*cm, 10*cm))
for i, phase in enumerate(phases):
    img_low = df_for_plotting.loc[phase, "low_map"]
    img_high = df_for_plotting.loc[phase, "high_map"]
    img_diff = df_for_plotting.loc[phase, "difference_map"]
    img_low[mask_combined==0] = np.nan
    img_high[mask_combined==0] = np.nan
    img_diff[mask_combined==0] = np.nan
    ax[i,0].imshow(img_low, vmin=0, vmax=0.8, cmap="viridis")
    im_hl = ax[i,1].imshow(img_high, vmin=0, vmax=0.8, cmap="viridis")
    im_diff = ax[i,2].imshow(img_diff, vmin=-0.3, vmax=0.3, cmap="PiYG")
    for a in ax[i,:]:
        overlay_allen(
            ax=a,
            areas_to_overlay=areas_to_overlay,
            show_bregma=False,
            res=(h, w),
            line_kw=dict(color="w", linewidth=0.5, alpha=0.5),
        )
    plt.colorbar(im_hl, ax=ax[i,1], fraction=0.046, pad=0.04)
    plt.colorbar(im_diff, ax=ax[i,2], fraction=0.046, pad=0.04)

    df_sig = stat_contrasts.query("p_adj < 0.05 and p== @phase")
    for idx, row in df_sig.iterrows():
        area_to_star = row["roi"].replace("_contra", "_R").replace("_ipsi", "_L")
        mask_roi = masks[area_names == area_to_star].squeeze()
        if mask_roi.sum() == 0:
            continue
        com = center_of_mass(mask_roi)
        pval = row["p_adj"]
        if pval < 0.001:
            star = "***"
        elif pval <= 0.01:
            star = "**"
        else:
            star = "*"
        ax[i,2].text(com[1], com[0], star, color="k", fontsize=fontsize_small - 1, ha="center", va="top")

plt.show()
# save
f.savefig(p_figures / 'rest_pupil_maps_low_vs_high.png', transparent=True, dpi=300)
f.savefig(p_figures / 'rest_pupil_maps_low_vs_high.svg', transparent=True, dpi=300)

# convert roi names to more readable format and save
stat_contrasts["roi"] = stat_contrasts["roi"].str.replace("_contra", "_R").str.replace("_ipsi", "_L")
stat_contrasts["Panel"] = "sf_revisions_pupil_B"
stat_contrasts["Statistical test"] = "Linear mixed-effects model"
stat_contrasts["Comparison"] = "low - high df/f %"
stat_contrasts["Post hoc test / multiple comparisons"] = "False discovery rate (Benjamini-Hochberg)"
stat_contrasts = stat_contrasts.rename(columns={
    "estimate": "Effect size",
    "p.value": "P-value",
    "confints": "95% Confidence intervals",
    "p_adj": "Adj. P-value",
    "p": "Stroke phase",
    "roi": "ROI",
}).filter(["Panel", "Statistical test", "Stroke phase", "ROI", "Comparison", "Effect size", "95% Confidence intervals", "P-value", "Post hoc test / multiple comparisons", "Adj. P-value"])
stat_contrasts.to_csv(p_figures / 'rest_pupil_maps_low_vs_high.csv', index=False)
